# IT135IU – D2: Data and profiling
**Group 06 – Topic #6:** Which listings on a room-rental platform are overpriced for their area?

Section 1 and 2 below are written by the Data Engineer. Other sections (data profile, cleaning log, data operations) are added by the other members.

## 1. Data source, loading and column selection (Data Engineer)

- **Dataset:** Sicily Airbnb Listings Dataset
- **Publisher:** Inside Airbnb – https://insideairbnb.com/get-the-data/
- **License:** CC BY 4.0 (attribution to Inside Airbnb required)
- **Date of retrieval:** 27 September 2026
- **Snapshot date:** 1 to 2 July 2026 (from `last_scraped`)
- **Files used:** `listings.csv` (detailed listings, 90 columns) and `reviews.csv` (summary reviews, 2 columns), both from the same snapshot

The data files are **not** included in this repository because Inside Airbnb does not allow redistribution. Download the files once from the link above, unzip them, and place them in the same folder as this notebook.

Only the 14 columns needed for the price comparison are loaded. Host-identifying columns are never read into this notebook.

In [4]:
import pandas as pd

# Random seed for reproducibility
RANDOM_SEED = 42

# 14 columns needed for the price comparison
selected_cols = [
    "id", "price", "neighbourhood_cleansed", "room_type", "property_type",
    "accommodates", "bedrooms", "bathrooms_text", "latitude", "longitude",
    "amenities", "number_of_reviews", "review_scores_rating", "minimum_nights",
]

# Read only these columns from the file (the other 76 columns are skipped)
df = pd.read_csv("listings.csv", usecols=selected_cols)

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Rows: 56873
Columns: 14


In [5]:
# Snapshot dates of the original file
print(pd.read_csv("listings.csv", usecols=["last_scraped"])["last_scraped"].unique())

<StringArray>
['2026-07-01', '2026-07-02']
Length: 2, dtype: str


**Result:** the working dataset has 56,873 rows and 14 columns (76 of the 90 original columns excluded). No rows were removed in this step.

**Columns excluded and why (also recorded in the cleaning log):**
- 10 host-identifying columns (names, IDs, profile URLs, descriptions, locations, pictures) were excluded for privacy, because the project does not need to identify hosts.
- 18 other host-related columns, including `host_is_superhost`, were excluded because the analysis focuses on listing prices and property characteristics, not on hosts. `host_is_superhost` is an evaluative label about an individual host, so it was removed to stay consistent with the commitment not to profile or evaluate hosts.
- The remaining excluded columns (listing URLs and descriptions, availability, extra review metrics, price-quote fields and similar) are not needed for the initial price comparison.

This step only selects columns. Missing values, price format and outliers are handled in the cleaning section below.

## 2. Merging the review table (Data Engineer)

A second table, `reviews.csv` from Inside Airbnb, is merged into the working dataset. It has two columns (`listing_id`, `date`) and one row per review. The detailed review file (with reviewer names, IDs and comment text) is **not** used, because the project does not need personal data of guests.

Steps: (1) summarise the reviews to one row per listing (count, first and latest review date), (2) merge into the working dataset by listing id with a left join, so no listing is lost, (3) check that both files come from the same snapshot.

In [6]:
# Table 2: reviews (only listing_id and date, no reviewer data)
reviews = pd.read_csv("reviews.csv", parse_dates=["date"])
print("Review rows:", len(reviews))
print("Date range:", reviews["date"].min().date(), "to", reviews["date"].max().date())

# One row per listing: number of reviews, date of the first and the latest review
review_summary = (
    reviews.groupby("listing_id")
    .agg(reviews_count=("date", "size"),
         first_review_date=("date", "min"),
         last_review_date=("date", "max"))
    .reset_index()
)
print("Listings with at least one review:", len(review_summary))

Review rows: 1301042
Date range: 2009-08-26 to 2026-07-02
Listings with at least one review: 40373


In [7]:
# Merge: keep all listings (left join), add the review summary by listing id
df_merged = df.merge(review_summary, left_on="id", right_on="listing_id",
                     how="left", validate="one_to_one")
df_merged = df_merged.drop(columns="listing_id")

print("Rows before merge:", len(df), "| after merge:", len(df_merged))

# A listing with no row in reviews.csv has no reviews
df_merged["reviews_count"] = df_merged["reviews_count"].fillna(0).astype(int)

# Checks: both files must come from the same snapshot
print("Review rows whose listing is not in the working dataset:",
      (~reviews["listing_id"].isin(df["id"])).sum())
print("Listings where reviews_count differs from number_of_reviews:",
      (df_merged["reviews_count"] != df_merged["number_of_reviews"]).sum())

df_merged.head(3)

Rows before merge: 56873 | after merge: 56873
Review rows whose listing is not in the working dataset: 14
Listings where reviews_count differs from number_of_reviews: 0


,id,neighbourhood_cleansed,latitude,longitude,property_type,room_type,accommodates,bathrooms_text,bedrooms,amenities,price,minimum_nights,number_of_reviews,review_scores_rating,reviews_count,first_review_date,last_review_date
0,7011,Marsala,37.839690,12.479490,Entire home,Entire home/apt,6,3 baths,3.0,"[""Hair dryer"", ""Single level home"", ""Long term...",$262.00,2.0,32,4.69,32,2009-08-26,2024-08-08
1,22948,Agrigento,37.308615,13.587409,Private room in bed and breakfast,Private room,2,1 shared bath,NaN,"[""Valley view"", ""Hair dryer"", ""Extra pillows a...",$53.58,1.0,507,4.83,507,2011-08-30,2026-06-09
2,35264,Piedimonte Etneo,37.824610,15.158760,Farm stay,Entire home/apt,14,3 baths,3.0,"[""Valley view"", ""Indoor fireplace"", ""Hair drye...",$534.00,1.0,168,4.81,168,2010-07-26,2026-02-01


**Merge decisions (for the cleaning log):**
- Join type: left join on `id` = `listing_id`, so all 56,873 listings are kept. The number of rows is the same before and after the merge.
- `validate="one_to_one"` makes pandas stop if either side has duplicate ids.
- A listing with no row in `reviews.csv` has no reviews, so `reviews_count` was set to 0 for these listings. `first_review_date` and `last_review_date` stay empty for them.
- `reviews.csv` has 14 review rows for 4 listings that are not in the working dataset. They are not used by the left join.
- Check: for every listing, `reviews_count` equals the original `number_of_reviews`, which confirms that both files come from the same snapshot.